# Phase 3 — LoRA reasoning SFT

A thin wrapper. Every decision lives in `configs/` and in `smolqwen.training.*`;
this notebook exists because Colab is where the GPU is, not because the logic
belongs in a notebook. If you find yourself editing a hyperparameter here rather
than in `configs/base/sft.yaml`, the run stops being reproducible from the repo.

Order matters in the setup cell: `scripts/setup_colab.sh` asserts the resolved
torch is 2.13.0 *before* building `flash-attn` and `causal-conv1d`, which compile
against the installed torch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!git clone https://github.com/qninhdt/smolqwen.git 2>/dev/null || true
%cd smolqwen
!bash scripts/setup_colab.sh

In [ ]:
# Capability only -- the throughput numbers come from this phase's own sweep,
# not from the probe.
!smolqwen probe

In [ ]:
import os
from getpass import getpass

# Prompted rather than hardcoded: a token pasted into a committed notebook is a
# leaked token. Both are optional -- absent credentials degrade to a local,
# untracked run rather than crashing.
for name in ("HF_TOKEN", "WANDB_API_KEY"):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name} (blank to skip): ")

In [ ]:
# Resolve and print the config without touching the GPU. `profile.max_seq_length`
# here must equal the cap in artifacts/data/budgets.json -- if it does not, a
# profile YAML is overriding a measured value.
!smolqwen train-sft --profile l4 --dry-run

In [ ]:
# One-time, ~20 minutes over the 701 MB release file. Skip if
# artifacts/data/sft/train.jsonl already exists.
!smolqwen profile-data
!smolqwen prepare-sft

In [ ]:
# A short run first: confirm the loss descends and no NaN appears with the
# toggles on together, before committing to the full schedule.
!smolqwen train-sft --profile l4 --override training.max_steps=30

In [ ]:
# The full run. `--resume` is safe to re-run after a VM reclaim: it continues
# from the pushed adapter and the same W&B run rather than forking the curve.
!smolqwen train-sft --profile l4
# !smolqwen train-sft --profile l4 --resume

In [ ]:
# Merge for Phase 5 eval and Phase 7 RL, then record the revision sha the merge
# consumed -- Phase 5 pins it.
!smolqwen merge-adapter --profile l4
!cat artifacts/models/qwen3.5-2b-sft-merged/merge_report.json